# RoboVAC: train a dirt detector (YOLO) on simulated floor images

Input: `dirt_yolo.zip` made by `tools/dirt_dataset/package_dataset.py` (images + YOLO boxes from the Gazebo floor camera).
Output: a trained `best.pt` plus ONNX / ncnn exports for a small board.

Runtime: **Runtime > Change runtime type > GPU** (a free T4 is plenty for `yolo11n`).

In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

## 1. Upload the dataset
Use the file panel on the left (upload `dirt_yolo.zip`), or mount Google Drive and point `ZIP` at it.

In [ ]:
import os, zipfile
ZIP = '/content/dirt_yolo.zip'      # change if you put it on Drive: /content/drive/MyDrive/dirt_yolo.zip
DATA = '/content/dirt_yolo'
assert os.path.exists(ZIP), 'upload dirt_yolo.zip first'
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DATA)
print(open(f'{DATA}/data.yaml').read())
import json; print(json.dumps(json.load(open(f'{DATA}/dataset_report.json')), indent=1)[:1500])

## 2. Look at the labels before training
If the boxes do not sit on the dirt here, stop: training on bad labels only wastes time.

In [ ]:
import glob, random, cv2, matplotlib.pyplot as plt
files = [f for f in glob.glob(f'{DATA}/images/train/*.jpg') if os.path.getsize(f.replace('images','labels').replace('.jpg','.txt')) > 0]
random.seed(0)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), random.sample(files, min(6, len(files)))):
    im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    for ln in open(f.replace('images', 'labels').replace('.jpg', '.txt')):
        c, cx, cy, bw, bh = map(float, ln.split())
        x0, y0 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        cv2.rectangle(im, (x0, y0), (x0 + int(bw * w), y0 + int(bh * h)), (255, 0, 0), 2)
    ax.imshow(im); ax.axis('off')
plt.show()

## 3. Train
Settings chosen for this data: no vertical flip and no rotation (the camera always looks down at the floor ahead, so
"up" in the picture is far away), colour/brightness jitter on (floors and lighting differ in real homes).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=f'{DATA}/data.yaml', imgsz=640, epochs=60, batch=16, patience=15,
            flipud=0.0, fliplr=0.5, degrees=0.0, hsv_h=0.02, hsv_s=0.6, hsv_v=0.4, mosaic=1.0,
            project='/content/runs', name='dirt', exist_ok=True)

## 4. Score on the held-out test episodes
These are floors and layouts the model never saw during training.

In [ ]:
best = YOLO('/content/runs/dirt/weights/best.pt')
m = best.val(data=f'{DATA}/data.yaml', split='test', imgsz=640)
print('precision', m.box.mp, 'recall', m.box.mr, 'mAP50', m.box.map50, 'mAP50-95', m.box.map)

## 5. The test that matters: real photos
A model that is only good on simulated floors is useless. Put 20-50 photos of real floors with real dirt
(taken at a low angle, about 7 cm high, like the robot camera) in a folder and upload it as `real_photos/`.
Expect it to do worse than on the simulator; how much worse is the result you report.

In [ ]:
REAL = '/content/real_photos'
if os.path.isdir(REAL):
    best.predict(source=REAL, imgsz=640, conf=0.25, save=True, project='/content/runs', name='real', exist_ok=True)
    for f in sorted(glob.glob('/content/runs/real/*.jpg'))[:8]:
        plt.figure(figsize=(8, 6)); plt.imshow(plt.imread(f)); plt.axis('off')
else:
    print('no real_photos/ folder yet; upload one to see how the model does outside the simulator')

## 6. Export for the robot's board
ONNX works almost everywhere (ONNX Runtime); ncnn is a good fit for ARM boards such as a Raspberry Pi.

In [ ]:
best.export(format='onnx', imgsz=640, simplify=True)
best.export(format='ncnn', imgsz=640)
from google.colab import files
files.download('/content/runs/dirt/weights/best.pt')